<a href="https://colab.research.google.com/github/joanby/python-ml-course/blob/master/notebooks/T5%20-%202%20-%20Logistic%20Regression%20-%20Implementación-Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clonamos el repositorio para obtener los dataSet

In [1]:
!git clone https://github.com/DavidArroyoTorres/python-ml-course/

Cloning into 'python-ml-course'...
remote: Enumerating objects: 17912, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 17912 (delta 100), reused 6 (delta 6), pack-reused 17758 (from 2)
Receiving objects: 100% (17912/17912), 531.92 MiB | 18.48 MiB/s, done.
Resolving deltas: 100% (440/440), done.
Updating files: 100% (16940/16940), done.


# Damos acceso a nuestro Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Test it
!ls '/content/drive/My Drive'

# Implementación el método de la máxima verosimilitud para la regresión logística

### Definir la función de entorno L(b)

In [50]:
from IPython.display import display, Math, Latex
display(Math(r'L(p;Y)=\prod_{i=1}^n p_i^{y_i}(1-p_i)^{1-y_i}, p_i=P(Y_i=1|X_{i,1}=x_{i,1},...,X_{i,k}=x_{i,k}) \text{ es  la función de verosimilitud de Y|X.}'))

<IPython.core.display.Math object>

In [51]:
def likelihood(y, pi):
    import numpy as np
    prod = 1
    prod_in = list(range(len(y)))
    for i in range(len(y)):
        prod_in[i] = np.where(y[i]==1, pi[i], 1-pi[i])
        prod = prod * prod_in[i]
    return prod

### Calcular las probabilidades para cada observación

In [52]:
display(Math(r'P_i = P(x_i) = \frac{1}{1+e^{-\beta_0-\sum_{j=1}^k\beta_j\cdot x_{ij}}} '))

<IPython.core.display.Math object>

In [ ]:
def logitprobs(X,beta): #Luego se ve el sentido, X=Matriz de tamaño nx(k+1) de los datos predictivos, beta vector de dimensión k+1.
    import numpy as np
    n_rows = np.shape(X)[0] #n=nº filas
    n_cols = np.shape(X)[1] #k+1=nº columnas
    p_i=list(range(n_rows)) #[0,..., n-1], con el mismo propósito se podría usar [0,...,0] con n ceros.
    expon=list(range(n_rows)) #[0,..., n-1] con el mismo propósito se podría usar [0,...,0] con n ceros.
    for i in range(n_rows): #[0,..., n-1]
        expon[i] = 0
        for j in range(n_cols): #[0,..., k]
            ex=X[i][j] * beta[j]
            expon[i] = ex + expon[i]
        with np.errstate(divide="ignore", invalid="ignore"):
            p_i[i]=1/(1+np.exp(-expon[i]))
    return p_i

### Calcular la matriz diagonal W

In [5]:
display(Math(r'W= diag(P_i \cdot (1-P_i))_{i=1}^n'))

<IPython.core.display.Math object>

In [ ]:
def findW(p_i):
    import numpy as np
    n = len(p_i)
    W = np.zeros(n*n).reshape(n,n)
    for i in range(n):
        print(i)
        W[i,i]=p_i[i]*(1-p_i[i])
        W[i,i].astype(float)
    return W

### Obtener la solución de la función logística

In [6]:
display(Math(r"\beta_{n+1} = \beta_n -\frac{f(\beta_n)}{f'(\beta_n)}"))
display(Math(r"f(\beta) = X(Y-P)"))
display(Math(r"f'(\beta) = XWX^T"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
def logistics(X, Y, limit):
    import numpy as np
    from numpy import linalg
    nrow = np.shape(X)[0]
    bias = np.ones(nrow).reshape(nrow,1)
    X_new = np.append(X, bias, axis = 1)
    ncol = np.shape(X_new)[1]
    beta = np.zeros(ncol).reshape(ncol,1)
    root_dif = np.array(range(1,ncol+1)).reshape(ncol,1)
    iter_i = 10000
    while(iter_i>limit):
        print("Iter:i"+str(iter_i) + ", limit:" + str(limit))
        p_i = logitprobs(X_new, beta)
        print("P_i:"+str(p_i))
        W = findW(p_i)
        print("W:"+str(W))
        num = (np.transpose(np.matrix(X_new))*np.matrix(Y - np.transpose(p_i)).transpose())
        den = (np.matrix(np.transpose(X_new))*np.matrix(W)*np.matrix(X_new))
        root_dif = np.array(linalg.inv(den)*num)
        beta = beta + root_dif
        print("Beta: "+str(beta))
        iter_i = np.sum(root_dif*root_dif)
        ll = likelihood(Y, p_i)
    return beta

## Comprobación experimental

In [ ]:
import numpy as np

In [ ]:
X = np.array(range(10)).reshape(10,1)

In [ ]:
X

array([[0],
       [1],
       [2],
       [3],
       [4],
       [5],
       [6],
       [7],
       [8],
       [9]])

In [ ]:
Y = [0,0,0,0,1,0,1,0,1,1]

In [ ]:
bias = np.ones(10).reshape(10,1)
X_new = np.append(X,bias,axis=1)

In [ ]:
X_new

array([[0., 1.],
       [1., 1.],
       [2., 1.],
       [3., 1.],
       [4., 1.],
       [5., 1.],
       [6., 1.],
       [7., 1.],
       [8., 1.],
       [9., 1.]])

In [ ]:
a = logistics(X,Y,0.00001)

Iter:i10000, limit:1e-05
Pi:[array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5])]
0
1
2
3
4
5
6
7
8
9
W:[[0.25 0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.25 0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.25 0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.25 0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.25 0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.25 0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.25 0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.25 0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.25 0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.25]]
Beta: [[ 0.43636364]
 [-2.36363636]]
Iter:i5.777190082644626, limit:1e-05
Pi:[array([0.08598797]), array([0.12705276]), array([0.18378532]), array([0.2583532]), array([0.35019508]), array([0.45467026]), array([0.56329497]), array([0.66616913]), array([0.75533524]), array([0.826

In [ ]:
ll = likelihood(Y, logitprobs(X,a))

In [ ]:
ll

array([1.32622426e-06])

In [ ]:
Y = 0.66220827 * X -3.69557172

# Con el paquete statsmodel de python

In [ ]:
import statsmodels.api as sm
import pandas as pd
from pandas import Timestamp

In [ ]:
Y = (Y - np.min(Y))/np.ptp(Y)
logit_model = sm.Logit(Y,X_new)

In [ ]:
result = logit_model.fit()

Optimization terminated successfully.
         Current function value: 0.359693
         Iterations 6


In [ ]:
print(result.summary2())

                         Results: Logit
Model:              Logit            Pseudo R-squared: 0.481    
Dependent Variable: y                AIC:              11.1939  
Date:               2020-09-19 17:21 BIC:              11.7990  
No. Observations:   10               Log-Likelihood:   -3.5969  
Df Model:           1                LL-Null:          -6.9315  
Df Residuals:       8                LLR p-value:      0.0098099
Converged:          1.0000           Scale:            1.0000   
No. Iterations:     6.0000                                      
------------------------------------------------------------------
           Coef.    Std.Err.      z      P>|z|     [0.025   0.975]
------------------------------------------------------------------
x1         0.6272     0.3735    1.6793   0.0931   -0.1048   1.3592
const     -2.8224     1.8730   -1.5069   0.1318   -6.4934   0.8485

